# FastConformer fine-tune - bản theo main

Notebook này là bản orchestration theo cấu trúc main: build code dataset, push kernel Kaggle chạy `asr_lab.train.finetune_vivos`, poll/pull artifact, đọc WER và xuất manifest. Notebook không nhồi logic train; logic nằm trong `src/asr_local` và main repo `asr_lab`.


## Phương pháp

Bản này dùng FastConformer Transducer 115M, thay vocabulary tiếng Việt rồi fine-tune full encoder và decoder RNNT trên VIVOS. Đây là hướng chuẩn hiện tại trong repo main cho FastConformer; chưa dùng LoRA/adapter hay freeze encoder.


In [ ]:
from pathlib import Path
import sys


def find_local_root(start=None):
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "asr_local").is_dir():
            return candidate
    raise RuntimeError("Không tìm thấy ASR_local/src/asr_local")


LOCAL_ROOT = find_local_root()
LOCAL_SRC = LOCAL_ROOT / "src"
if str(LOCAL_SRC) not in sys.path:
    sys.path.insert(0, str(LOCAL_SRC))

from asr_local.paths import bootstrap

LOCAL_ROOT, MAIN_ROOT = bootstrap(LOCAL_ROOT)
print("LOCAL_ROOT =", LOCAL_ROOT)
print("MAIN_ROOT  =", MAIN_ROOT)


In [ ]:
from dataclasses import asdict

from IPython.display import display

from asr_local.analytics.compare import (
    artifact_manifest,
    load_results,
    result_table,
    run_main_report_commands,
)
from asr_local.data.vivos import prepare_vivos_manifests
from asr_local.deploy.kaggle import (
    build_code_dataset,
    poll_kernel,
    pull_kernel,
    push_common_voice_resume,
    push_vivos_finetune,
    run_cmd,
    upload_resume_dataset,
    uv_module,
)
from asr_local.model.fastconformer import CommonVoiceRunConfig, FastConformerRunConfig


## Cấu hình run final theo main


In [ ]:
RUN_REMOTE = False

config = FastConformerRunConfig(
    account="trnmnhtn",
    kernel="asr-ft-fc115m-v2norm",
    run_id="vivos-fc115m-v2norm",
    model="nvidia/stt_en_fastconformer_transducer_large",
    epochs=50,
    batch=16,
    vocab_size=1024,
    lr="2e-4",
    precision="32",
    max_minutes=480,
)

display(asdict(config))
print("script args:")
print(config.script_args())


## Kiểm tra account Kaggle


In [ ]:
run_cmd(
    uv_module("asr_lab.deploy.kaggle", "accounts"),
    MAIN_ROOT,
    RUN_REMOTE,
)


## Build code dataset


In [ ]:
build_code_dataset(config, MAIN_ROOT, RUN_REMOTE)


## Push kernel fine-tune VIVOS


In [ ]:
push_vivos_finetune(config, MAIN_ROOT, RUN_REMOTE)


## Poll kernel


In [ ]:
poll_kernel(config.account, config.kernel, MAIN_ROOT, RUN_REMOTE)


## Pull artifact về main repo


In [ ]:
pull_kernel(config.account, config.kernel, MAIN_ROOT, RUN_REMOTE)


## Đọc results.json


In [ ]:
results = load_results(MAIN_ROOT, config.run_id)
display(result_table(results))


## Report và compare theo main


In [ ]:
BASE_RUN_ID = "vivos-fc115m-v1"
RUN_MAIN_REPORTS = False

if RUN_MAIN_REPORTS:
    run_main_report_commands(MAIN_ROOT, BASE_RUN_ID, config.run_id)
else:
    print("Đặt RUN_MAIN_REPORTS=True khi đã có cả baseline và candidate results.json.")


## Manifest artifact


In [ ]:
display(artifact_manifest(MAIN_ROOT, config.run_id))


## Optional: resume sang Common Voice tiếng Việt


In [ ]:
cv_config = CommonVoiceRunConfig(
    account=config.account,
    kernel="asr-cv-fc115m",
    run_id="vivos-cv",
    resume_dataset="asr-v2norm-nemo",
    epochs=25,
    batch=16,
    precision="32",
)

nemo_path = cv_config.resume_nemo(MAIN_ROOT, config.run_id)
display(asdict(cv_config))
print("resume .nemo:", nemo_path)
print("script args:")
print(cv_config.script_args())


In [ ]:
RUN_COMMON_VOICE = False

if RUN_COMMON_VOICE:
    upload_resume_dataset(cv_config, nemo_path, MAIN_ROOT, RUN_REMOTE)
    push_common_voice_resume(cv_config, MAIN_ROOT, RUN_REMOTE)
    poll_kernel(cv_config.account, cv_config.kernel, MAIN_ROOT, RUN_REMOTE)
    pull_kernel(cv_config.account, cv_config.kernel, MAIN_ROOT, RUN_REMOTE)
else:
    print("Bỏ qua Common Voice resume trong bản chạy VIVOS final.")


## Checklist trước khi chạy thật

- Đã có `~/.kaggle/trnmnhtn/kaggle.json`.
- Đặt `RUN_REMOTE = True` sau khi đã xem dry-run commands.
- Không đổi `run_id` final nếu muốn artifact nằm đúng `artifacts/runs/vivos-fc115m-v2norm`.
